# 06. Tổng kết (Bước 8)

**Nhiệm vụ của notebook này:** gom toán dùng ở đâu, trả lời bốn câu hỏi của thầy và nói thẳng hạn chế (mục 2.8).

Chạy từng ô từ trên xuống. Notebook tự đọc `data.csv` (Colab sẽ hỏi bạn tải lên; trên máy thì đặt tại `data/data.csv`).

Nên đọc theo thứ tự các notebook 01 → 06. Phần trước: 05.

---
# Bước 8. Tổng kết

## Toán nào dùng ở đâu
| Mảng toán | Dùng ở đâu | Bằng chứng trong notebook |
|---|---|---|
| Đại số tuyến tính | Vector, ma trận, tích vô hướng, Q·K, trị riêng, số điều kiện | Bước 2, 7 |
| Giải tích | Sigmoid, cross-entropy, gradient (p − y), quy tắc dây chuyền | Bước 3, 4 (bản tự cài khớp sklearn) |
| Tối ưu | Gradient Descent, tốc độ học, tiêu chuẩn dừng; Muon (ở TabICLv2) | Bước 4 (bảng 4 tốc độ học × chuẩn hóa) |
| Xác suất, thống kê | P(y\|x), hợp lý cực đại, chia train/test, overfitting, sai số chuẩn | Bước 3, 5 |
| Đánh giá | Ma trận nhầm lẫn, Recall, ngưỡng | Bước 6 |

## Hạn chế cần nói thẳng
1. **TabICLv2 mới chạy một lần**, cấu hình mặc định, một lần chia dữ liệu; chưa khảo sát kích thước ngữ cảnh. Thí nghiệm tối ưu vẫn làm trên Hồi quy Logistic. Trên cùng cách chia, TabICLv2 có AUC cao hơn ở cả bốn cấu hình nhưng chênh Accuracy (0,1 đến 1,4 điểm) nhỏ hơn sai số ±2,2 điểm.
2. **Điểm HK2 phần nào đã chứa sẵn đáp án**; cảnh báo sớm thật sự dựa vào B1.
3. **Hai cột "môn qua HK1" và "môn qua HK2" tương quan khoảng 0,92** (cộng tuyến), nên không nên đọc riêng từng trọng số.
4. **Một lần chia dữ liệu**, sai số Accuracy cỡ ±2,2 điểm; chưa kiểm tra chéo.
5. Dữ liệu chỉ từ một trường ở Bồ Đào Nha. Dự đoán nguy cơ chỉ nên dùng để *hỗ trợ* sinh viên, không để quy kết.

## Trả lời bốn câu hỏi của thầy
1. **Mô hình hóa:** phân loại nhị phân; mỗi sinh viên là một vector 8 chiều; đầu ra là xác suất bỏ học.
2. **Toán quyết định cách hoạt động:** đại số tuyến tính để biểu diễn và gộp thông tin, giải tích để đo sai lệch và chỉ hướng sửa, tối ưu để thực sự tìm ra tham số, xác suất thống kê để biết có tin được trên dữ liệu mới không.
3. **Tham số học bằng cách:** cực tiểu hóa cross-entropy bằng Gradient Descent (mô hình đối chứng); TabICLv2 được huấn luyện trước một lần bằng Muon rồi dùng không cần cập nhật.
4. **Giải thích kết quả:** số điều kiện giải thích vì sao phải chuẩn hóa; cross-entropy và ngưỡng giải thích cách cân giữa bỏ sót và báo nhầm.

**Rút ra:** đoán đúng hay sai chưa đáng giá bằng việc biết cái giá của từng kiểu sai. Toán cho ta công cụ để đo cái giá đó.

In [1]:
# Ô thiết lập: nạp thư viện và đọc dữ liệu
import io, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

warnings.filterwarnings("ignore")
SEED = 42                               # hạt giống ngẫu nhiên cố định: chạy lại ra đúng kết quả cũ
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def doc_du_lieu():
    """Tìm data.csv trên máy; nếu đang ở Colab thì xin người dùng tải lên."""
    for p in ["data/data.csv", "../data/data.csv", "data.csv"]:
        if os.path.exists(p):
            print("Đọc dữ liệu từ", p)
            return pd.read_csv(p, sep=";")
    try:
        from google.colab import files
        print("Hãy chọn tệp data.csv trên máy của bạn ...")
        up = files.upload()
        return pd.read_csv(io.BytesIO(list(up.values())[0]), sep=";")
    except ImportError:
        raise FileNotFoundError("Không thấy data.csv. Hãy đặt tệp vào thư mục data/.")


raw = doc_du_lieu()
raw.columns = [c.strip() for c in raw.columns]      # bỏ khoảng trắng/tab thừa ở tên cột
print("Kích thước:", raw.shape)

# --- Chuẩn bị chung (lặp lại ở mỗi notebook để mỗi tệp chạy độc lập) ---
d = raw[raw.Target != "Enrolled"].reset_index(drop=True)
d["y"] = (d.Target == "Dropout").astype(int)          # 1 = bỏ học (nguy cơ), 0 = tốt nghiệp (an toàn)
ENTRY = ["Tuition fees up to date", "Debtor", "Scholarship holder", "Age at enrollment"]
YEAR1 = ["Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
         "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)"]
COMPACT = ENTRY + YEAR1
ADMISSION_ONLY = ["Admission grade", "Previous qualification (grade)", "Age at enrollment",
                  "Scholarship holder", "Debtor", "Tuition fees up to date", "Gender", "Application order"]
ALL = [c for c in d.columns if c not in ("Target", "y")]
y = d.y.values
train_idx, test_idx = train_test_split(np.arange(len(d)), test_size=0.2, stratify=d.y, random_state=SEED)
ytr, yte = y[train_idx], y[test_idx]


def chuan_bi(cols, chuan_hoa=True):
    """Ma trận X cho tập huấn luyện và kiểm tra; chuẩn hóa bằng trung bình, độ lệch chuẩn của TẬP HUẤN LUYỆN."""
    X = d[cols].values.astype(float)
    Xtr, Xte = X[train_idx], X[test_idx]
    if chuan_hoa:
        mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd == 0] = 1
        Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    return Xtr, Xte


def hoi_quy_sklearn(X, y):
    try:
        return LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
    except Exception:
        return LogisticRegression(C=np.inf, max_iter=5000).fit(X, y)


def do_metrics(y_that, p, nguong=0.5):
    yh = (p >= nguong).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_that, yh).ravel()
    return {"accuracy": accuracy_score(y_that, yh), "precision": precision_score(y_that, yh, zero_division=0),
            "recall": recall_score(y_that, yh), "f1": f1_score(y_that, yh), "auc": roc_auc_score(y_that, p),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}


CAU_HINH = {"A. Lúc nhập học (8 cột)": ADMISSION_ONLY,
            "B1. Nhập học + hết HK1 (6 cột)": ENTRY + YEAR1[:2],
            "B. Nhập học + hết HK2 (8 cột)": COMPACT,
            "C. Đầy đủ 35 cột": ALL}


# Bảng tổng hợp: Hồi quy Logistic đối chứng trên bốn cấu hình (TabICLv2 thêm vào đây sau khi chạy ở notebook 05)
hang = []
for ten_ch, cot in CAU_HINH.items():
    Xa, Xb = chuan_bi(cot, True)
    p = hoi_quy_sklearn(Xa, ytr).predict_proba(Xb)[:, 1]
    hang.append({"cấu hình": ten_ch, **{k: v for k, v in do_metrics(yte, p).items() if k in ("accuracy", "recall", "auc")}})
display(pd.DataFrame(hang).round(3))

Đọc dữ liệu từ ../data/data.csv
Kích thước: (4424, 37)


,cấu hình,accuracy,recall,auc
0,A. Lúc nhập học (8 cột),0.751,0.581,0.815
1,B1. Nhập học + hết HK1 (6 cột),0.875,0.806,0.914
2,B. Nhập học + hết HK2 (8 cột),0.899,0.856,0.940
3,C. Đầy đủ 35 cột,0.945,0.923,0.973
